# Topic 30 · Forecasting the S&P 500, and the simplest baseline

CSCI 6379 · companion to the lecture note [Recurrent neural networks](https://dongchul.kim/csci6379-fall2026/topic/30).

An LSTM forecasts the S&P 500 from the last 30 days. First tomorrow's **price**, against the forecast that does nothing ("tomorrow = today"). Then, from the same data, the **size** of tomorrow's move.

**Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. It takes about 2 minutes on a T4.

In [ ]:
import csv, io, urllib.request
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt

# FRED series SP500, daily close (FRED keeps the last ten years). Days without a value are dropped.
URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=SP500&cosd=2016-01-01&coed=2025-12-31"
rows = [r for r in csv.reader(io.StringIO(urllib.request.urlopen(URL).read().decode())) if r[0][0].isdigit() and r[1] not in ("", ".")]
dates = np.array([r[0] for r in rows]); price = np.array([float(r[1]) for r in rows])
print(len(price), "days,", dates[0], "to", dates[-1])

class Forecaster(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, 32, batch_first=True)
        self.out = nn.Linear(32, 1)
    def forward(self, x):                      # x: (batch, 30 days, 1)
        h, _ = self.lstm(x)
        return self.out(h[:, -1]).squeeze(-1)

def fit(Xtr, Ytr, Xte, seed=0):
    torch.manual_seed(seed); net = Forecaster(); opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    xt, yt = torch.tensor(Xtr, dtype=torch.float32), torch.tensor(Ytr, dtype=torch.float32)
    for epoch in range(60):
        perm = torch.randperm(len(xt))
        for i in range(0, len(xt), 64):
            b = perm[i:i + 64]; loss = nn.functional.mse_loss(net(xt[b]), yt[b])
            opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return net(torch.tensor(Xte, dtype=torch.float32)).numpy()

## 1. Tomorrow's price

Each 30-day window is divided by its last close, so the model sees the shape of the month rather than the level of the index. Train on 2016 to 2022, test on 2023 to 2025, in time order.

In [ ]:
W = 30
X, Y, idx = [], [], []
for t in range(W - 1, len(price) - 1):
    win = price[t - W + 1:t + 1]
    X.append((win / price[t] - 1) * 100); Y.append((price[t + 1] / price[t] - 1) * 100); idx.append(t)
X = np.array(X)[:, :, None]; Y = np.array(Y); idx = np.array(idx)
split = np.searchsorted(dates[idx + 1], "2023-01-01")
r = fit(X[:split], Y[:split], X[split:])
today, actual = price[idx[split:]], price[idx[split:] + 1]
pred = today * (1 + r / 100)
rmse = lambda p: np.sqrt(np.mean((p - actual) ** 2))
print(f"error (RMSE): LSTM {rmse(pred):.1f}   'tomorrow = today' {rmse(today):.1f}")
print(f"direction right: LSTM {np.mean(np.sign(r) == np.sign(actual - today)):.1%}   'always up' {np.mean(actual > today):.1%}")

d = dates[idx[split:] + 1].astype("datetime64[D]")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(d, actual, label="actual"); ax[0].plot(d, pred, label="LSTM"); ax[0].legend(); ax[0].set_title("looks almost perfect")
z = slice(-60, None)
ax[1].plot(d[z], actual[z], "o-", label="actual"); ax[1].plot(d[z], pred[z], "o-", label="LSTM")
ax[1].plot(d[z], today[z], "s--", label="tomorrow = today"); ax[1].legend(); ax[1].set_title("the last 60 days")
plt.show()

## 2. The size of tomorrow's move

Same data, same LSTM. The input is the last 30 daily returns, the target how big tomorrow's move is, whichever way.

In [ ]:
ret = np.diff(price) / price[:-1] * 100
X2, Y2, day = [], [], []
for t in range(W, len(ret)):
    X2.append(ret[t - W:t]); Y2.append(abs(ret[t])); day.append(dates[t + 1])
X2 = np.array(X2)[:, :, None]; Y2 = np.array(Y2); day = np.array(day)
s2 = np.searchsorted(day, "2023-01-01")
p2 = fit(X2[:s2], Y2[:s2], X2[s2:]); y2 = Y2[s2:]
mae = lambda p: np.mean(np.abs(p - y2))
print(f"error (MAE, percentage points): LSTM {mae(p2):.3f}")
print(f"  same as today {mae(np.abs(X2[s2:, -1, 0])):.3f}   training average {mae(np.full(len(y2), Y2[:s2].mean())):.3f}"
      f"   last 30 days' average {mae(np.abs(X2[s2:, :, 0]).mean(1)):.3f}")
plt.figure(figsize=(10, 3.4))
dd = day[s2:].astype("datetime64[D]")
plt.bar(dd, y2, width=1, color="lightgray", label="actual size of the move")
plt.plot(dd, p2, label="LSTM forecast"); plt.ylim(0, 4); plt.legend(); plt.show()